In [18]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')


In [ ]:
ACN_PATH       = '../data/raw/acn/acndata_sessions.json.xlsx'
ST_EVCDP_PATH  = '../data/raw/st_evcdp/'
PROCESSED_PATH = '../data/processed/'

os.makedirs(PROCESSED_PATH, exist_ok=True)

Ready


In [34]:
df_acn = pd.read_excel(ACN_PATH)


drop_cols = ['_meta', 'end', 'start', 'min_kWh', '_items', 'userID.1', 'modifiedAt']
df_acn.drop(columns=[c for c in drop_cols if c in df_acn.columns], inplace=True)


for col in ['connectionTime', 'disconnectTime', 'doneChargingTime']:
    df_acn[col] = pd.to_datetime(df_acn[col], utc=True, errors='coerce')


before = len(df_acn)
df_acn.dropna(subset=['connectionTime', 'disconnectTime', 'kWhDelivered'], inplace=True)
print(f"Dropped {before - len(df_acn)} rows")


df_acn = df_acn[df_acn['kWhDelivered'] > 0]


df_acn['session_duration_hrs'] = (
    df_acn['disconnectTime'] - df_acn['connectionTime']
).dt.total_seconds() / 3600

df_acn = df_acn[df_acn['session_duration_hrs'] > 0]
df_acn = df_acn[df_acn['session_duration_hrs'] < 24]

df_acn['hour_of_day']  = df_acn['connectionTime'].dt.hour
df_acn['day_of_week']  = df_acn['connectionTime'].dt.dayofweek  # 0=Monday
df_acn['is_weekend']   = df_acn['day_of_week'].isin([5, 6]).astype(int)
df_acn['month']        = df_acn['connectionTime'].dt.month
df_acn['date']         = df_acn['connectionTime'].dt.date


BASELINE_TARIFF = 15
df_acn['revenue_session']    = df_acn['kWhDelivered'] * BASELINE_TARIFF
df_acn['charging_rate_kw']   = df_acn['kWhDelivered'] / df_acn['session_duration_hrs']

print(f"ACN cleaned shape: {df_acn.shape}")
df_acn[['connectionTime','kWhDelivered','session_duration_hrs','hour_of_day','revenue_session']].head(3)

Dropped 1305 rows
ACN cleaned shape: (14848, 28)


,connectionTime,kWhDelivered,session_duration_hrs,hour_of_day,revenue_session
0,2018-04-25 11:08:04+00:00,7.932,2.201667,11,118.980
1,2018-04-25 13:45:10+00:00,10.013,11.185000,13,150.195
2,2018-04-25 13:45:50+00:00,5.257,9.315278,13,78.855


In [ ]:
occupancy = pd.read_csv(ST_EVCDP_PATH + 'occupancy.csv', index_col=0)
volume    = pd.read_csv(ST_EVCDP_PATH + 'volume.csv',    index_col=0)
duration  = pd.read_csv(ST_EVCDP_PATH + 'duration.csv',  index_col=0)
price     = pd.read_csv(ST_EVCDP_PATH + 'price.csv',     index_col=0)
time_df   = pd.read_csv(ST_EVCDP_PATH + 'time.csv')
stations  = pd.read_csv(ST_EVCDP_PATH + 'stations.csv')

print("occupancy :", occupancy.shape)
print("volume    :", volume.shape)
print("duration  :", duration.shape)
print("price     :", price.shape)
print("time      :", time_df.shape)
print("stations  :", stations.shape)

print("\nStation IDs (first 5):", occupancy.columns[:5].tolist())
print("\noccupancy sample:\n", occupancy.iloc[:3, :4])
print("\nprice sample:\n", price.iloc[:3, :4])

occupancy : (8640, 247)
volume    : (8640, 247)
duration  : (8640, 247)
price     : (8640, 247)
time      : (8640, 6)
stations  : (1706, 6)

Station IDs (first 5): ['102', '105', '107', '108', '109']

occupancy sample:
            102  105  107  108
timestamp                    
1           12   16   24   15
2           12   16   24   15
3           12   16   24   15

price sample:
              102       105       107   108
timestamp                                 
1          0.924  1.124167  0.926364  0.99
2          0.924  1.124167  0.926364  0.99
3          0.924  1.124167  0.926364  0.99


In [ ]:
timestamps = pd.to_datetime({
    'year':   time_df['year'],
    'month':  time_df['month'],
    'day':    time_df['day'],
    'hour':   time_df['hour'],
    'minute': time_df['minute'],
    'second': time_df['second']
})


min_rows = min(len(timestamps), len(occupancy))
timestamps        = timestamps.iloc[:min_rows]
occupancy_trimmed = occupancy.iloc[:min_rows].astype(float)
volume_trimmed    = volume.iloc[:min_rows].astype(float)
duration_trimmed  = duration.iloc[:min_rows].astype(float)
price_trimmed     = price.iloc[:min_rows].astype(float)


station_ids = occupancy_trimmed.columns.tolist()
n_times     = len(timestamps)
n_stations  = len(station_ids)

print(f"{n_times * n_stations:,} rows")

ts_repeated = np.repeat(timestamps.values, n_stations)
st_tiled    = np.tile(station_ids, n_times)

df_ev = pd.DataFrame({
    'timestamp'    : ts_repeated,
    'station_id'   : st_tiled,
    'occupancy'    : occupancy_trimmed.values.flatten(),
    'volume'       : volume_trimmed.values.flatten(),
    'duration_mins': duration_trimmed.values.flatten(),
    'price_per_kwh': price_trimmed.values.flatten(),
})

df_ev['timestamp'] = pd.to_datetime(df_ev['timestamp'])
df_ev['station_id'] = df_ev['station_id'].astype(str)


df_ev['hour_of_day'] = df_ev['timestamp'].dt.hour
df_ev['day_of_week'] = df_ev['timestamp'].dt.dayofweek
df_ev['is_weekend']  = df_ev['day_of_week'].isin([5,6]).astype(int)
df_ev['month']       = df_ev['timestamp'].dt.month
df_ev['date']        = df_ev['timestamp'].dt.date


stations['station_id'] = stations['station_id'].astype(str)
df_ev = df_ev.sort_values(['station_id','timestamp']).reset_index(drop=True)
df_ev = df_ev.merge(stations[['station_id','count']], on='station_id', how='left')


df_ev['utilization_rate'] = (df_ev['occupancy'] / df_ev['count']).clip(0, 1)

df_ev['congestion_flag'] = (df_ev['utilization_rate'] > 0.8).astype(int)
df_ev['low_demand_flag'] = (df_ev['utilization_rate'] < 0.3).astype(int)

print(f"Shape: {df_ev.shape}")
print(f"Date range: {df_ev['timestamp'].min()} to {df_ev['timestamp'].max()}")
print(f"\nValue ranges:")
print(f"  occupancy : {df_ev['occupancy'].min():.2f} to {df_ev['occupancy'].max():.2f}")
print(f"  volume    : {df_ev['volume'].min():.2f} to {df_ev['volume'].max():.2f}")
print(f"  price     : {df_ev['price_per_kwh'].min():.2f} to {df_ev['price_per_kwh'].max():.2f}")
print(f"  duration  : {df_ev['duration_mins'].min():.2f} to {df_ev['duration_mins'].max():.2f}")
df_ev.head(3)

Building 2,134,080 rows...
Shape: (2134080, 15)
Date range: 2022-06-19 00:00:00 to 2022-07-18 23:55:00

Value ranges:
  occupancy : 0.00 to 220.00
  volume    : 0.00 to 1492.50
  price     : 0.25 to 1.47
  duration  : 0.00 to 17.08


,timestamp,station_id,occupancy,volume,duration_mins,price_per_kwh,hour_of_day,day_of_week,is_weekend,month,date,count,utilization_rate,congestion_flag,low_demand_flag
0,2022-06-19 00:00:00,1000,60.0,16.197222,3.157778,0.894267,0,6,1,6,2022-06-19,10,1.0,1,0
1,2022-06-19 00:05:00,1000,60.0,24.791667,4.833333,0.894267,0,6,1,6,2022-06-19,10,1.0,1,0
2,2022-06-19 00:10:00,1000,60.0,24.791667,4.833333,0.894267,0,6,1,6,2022-06-19,10,1.0,1,0


In [ ]:
print("Missing values in ACN:")
acn_missing = df_acn.isnull().sum()
print(acn_missing[acn_missing > 0])

print("\nMissing values in ST-EVCDP:")
ev_missing = df_ev.isnull().sum()
print(ev_missing[ev_missing > 0])


df_ev['volume'].fillna(0, inplace=True)
df_ev['duration_mins'].fillna(df_ev['duration_mins'].median(), inplace=True)
df_ev['price_per_kwh'].fillna(df_ev['price_per_kwh'].median(), inplace=True)
df_ev['utilization_rate'].fillna(0, inplace=True)


df_ev['congestion_flag'] = (df_ev['utilization_rate'] > 0.8).astype(int)
df_ev['low_demand_flag'] = (df_ev['utilization_rate'] < 0.3).astype(int)

print("\nAfter fix:")
print(f"  utilization range : {df_ev['utilization_rate'].min():.2f} to {df_ev['utilization_rate'].max():.2f}")
print(f"  congestion %      : {df_ev['congestion_flag'].mean()*100:.1f}%")
print(f"  low demand %      : {df_ev['low_demand_flag'].mean()*100:.1f}%")


df_acn.to_csv(PROCESSED_PATH + 'acn_caltech_processed.csv', index=False)
df_ev.to_csv(PROCESSED_PATH + 'st_evcdp_processed.csv', index=False)

print("\nSaved:")
print(f"  acn_caltech_processed.csv  — {df_acn.shape}")
print(f"  st_evcdp_processed.csv     — {df_ev.shape}")

Missing values in ACN:
site                  14847
doneChargingTime          8
userID                12636
userInputs            14848
WhPerMile             12636
kWhRequested          12636
milesRequested        12636
minutesAvailable      12636
paymentRequired       12636
requestedDeparture    12636
dtype: int64

Missing values in ST-EVCDP:
Series([], dtype: int64)

After fix:
  utilization range : 0.00 to 1.00
  congestion %      : 66.1%
  low demand %      : 16.2%

Saved:
  acn_caltech_processed.csv  — (14848, 28)
  st_evcdp_processed.csv     — (2134080, 15)

Notebook 1 done! Ready for EDA.


In [ ]:
df_acn.to_csv(PROCESSED_PATH + 'acn_caltech_processed.csv', index=False)
df_ev.to_csv(PROCESSED_PATH + 'st_evcdp_processed.csv', index=False)

print(f"  acn_caltech_processed.csv  — {df_acn.shape}")
print(f"  st_evcdp_processed.csv     — {df_ev.shape}")


Saved:
  acn_caltech_processed.csv  — (14848, 28)
  st_evcdp_processed.csv     — (2134080, 15)

Notebook 1 done! Ready for EDA.
